# **0: Menyiapkan Tempat Penyimpanan (Google Drive)**

In [ ]:
# =====================================================================
# SESI 0: MOUNT GOOGLE DRIVE (PENYIMPANAN AMAN)
# =====================================================================
from google.colab import drive
import os

# 1. Hubungkan Colab ke Google Drive
print("Menghubungkan ke Google Drive...")
drive.mount('/content/drive')

# 2. Tentukan folder khusus agar rapi di Drive Anda
folder_output = "/content/drive/MyDrive/Tugas_RAG_GapMatrix"
os.makedirs(folder_output, exist_ok=True)

print(f"\n✅ Berhasil! Semua hasil akan disimpan di folder: {folder_output}")

Menghubungkan ke Google Drive...
Mounted at /content/drive

✅ Berhasil! Semua hasil akan disimpan di folder: /content/drive/MyDrive/Tugas_RAG_GapMatrix


# **1. Instalasi Tools, Library, dan Dependensi Sistem**

In [ ]:
# =====================================================================
# SESI 1: INSTALASI DEPENDENSI SISTEM & LIBRARY PYTHON
# =====================================================================

# 1. Update paket Linux Colab dan instal software Tesseract OCR + Bahasa pendukung
!sudo apt-get update -qq
!sudo apt-get install -y tesseract-ocr tesseract-ocr-ind tesseract-ocr-eng -qq

# 2. Instal seluruh library Python untuk pemrosesan dokumen, vektor database, dan embedding
!pip install -q pymupdf pytesseract pillow chromadb sentence-transformers openai


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


# **2. Fungsi Ekstraksi Dokumen PDF Secara Hybrid**

In [ ]:
# =====================================================================
# SESI 2: FUNGSI EKSTRAKSI PDF HYBRID (PyMuPDF + Tesseract OCR)
# =====================================================================
import io
import os
import pymupdf
import pytesseract
from PIL import Image

def baca_pdf_hybrid(path_file):
    """
    Membaca dokumen PDF. Jika teks digital tidak terbaca,
    sistem otomatis melakukan rendering halaman menjadi gambar dan menjalankan OCR.
    """
    if not os.path.exists(path_file):
        return f"Eror: File '{path_file}' tidak ditemukan."

    dokumen = pymupdf.open(path_file)
    teks_lengkap = ""

    print(f"Memulai pemrosesan dokumen: {path_file}")
    print(f"Total halaman: {len(dokumen)}")

    for indeks_hal, halaman in enumerate(dokumen):
        nomor_halaman = indeks_hal + 1
        teks_halaman = halaman.get_text()

        # Validasi: Jika teks kurang dari 10 karakter, asumsikan ini halaman scan/gambar
        if len(teks_halaman.strip()) < 10:
            print(f"  -> [Halaman {nomor_halaman}] Teks digital kosong. Menjalankan Tesseract OCR...")
            pix = halaman.get_pixmap(dpi=150)
            img_data = pix.tobytes("png")
            gambar = Image.open(io.BytesIO(img_data))

            # Membaca teks Bahasa Indonesia dan Inggris sekaligus
            teks_ocr = pytesseract.image_to_string(gambar, lang='ind+eng')
            teks_lengkap += teks_ocr + "\n"
        else:
            print(f"  -> [Halaman {nomor_halaman}] Berhasil mengekstrak teks asli.")
            teks_lengkap += teks_halaman + "\n"

    return teks_lengkap

# **3. Fungsi Pemotongan Teks Berbasis Kata**

In [ ]:
# =====================================================================
# SESI 3: FUNGSI CHUNKING BERBASIS KATA (Layout-Safe)
# =====================================================================

def chunking_akademik(teks, chunk_size=1000, overlap=200):
    # Split berdasarkan paragraf ganda, lalu kalimat
    import re
    paragraf = re.split(r'\n\s*\n', teks)
    chunks = []
    current = ""
    for p in paragraf:
        if len(current) + len(p) < chunk_size:
            current += p + "\n\n"
        else:
            if current:
                chunks.append(current.strip())
            current = p + "\n\n"
    if current:
        chunks.append(current.strip())
    return chunks


# **4. Inisialisasi Vektor Database & Penyimpanan Dokumen Berkelanjutan**

In [ ]:
# =====================================================================
# SESI 4: INISIALISASI CHROMADB & INGESTI DOKUMEN SECARA BERTAHAP
# =====================================================================
import chromadb
from chromadb.utils import embedding_functions

# 1. Konfigurasi direktori penyimpanan lokal database
path_database = "./chroma_db_rag"
client = chromadb.PersistentClient(path=path_database)

# 2. Registrasi model embedding gratis
model_embedding = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
)

# 3. Membuat atau membuka koleksi database RAG
koleksi = client.get_or_create_collection(
    name="koleksi_rag_pdf",
    embedding_function=model_embedding
)

# =====================================================================
# PENGATURAN INPUT: Tentukan nama file yang ingin diproses sekarang
# =====================================================================
nama_file_sekarang = "jurnal10.pdf" # <--- UBAH INI SETIAP GANTI JURNAL

print(f"--- JALUR INGESTI DATA UNTUK: {nama_file_sekarang} ---")

if not os.path.exists(nama_file_sekarang):
    print(f"⚠️ Eror: Silakan upload file '{nama_file_sekarang}' ke folder kiri Colab terlebih dahulu.")
else:
    # Langkah A: Ekstraksi teks melalui Sesi 2
    teks_mentah = baca_pdf_hybrid(nama_file_sekarang)

    # Langkah B: Segmentasi teks melalui Sesi 3 (PERBAIKAN: Gunakan chunking_akademik)
    daftar_chunk = chunking_akademik(teks_mentah, chunk_size=1000, overlap=200)
    print(f"✓ Berhasil memotong dokumen menjadi {len(daftar_chunk)} chunk.")

    # Langkah C: Pembuatan ID unik dan metadata penanda berkas
    ids_chunk = []
    metadata_chunk = []
    for indeks, chunk in enumerate(daftar_chunk):
        ids_chunk.append(f"{nama_file_sekarang}_chunk_{indeks}")
        metadata_chunk.append({"sumber": nama_file_sekarang})

    # Langkah D: Penyimpanan teks dan embedding ke ChromaDB
    print("-> Sedang memproses vektor embedding dan menyimpan ke database...")
    koleksi.add(
        documents=daftar_chunk,
        ids=ids_chunk,
        metadatas=metadata_chunk
    )

    print(f"🎉 SUKSES! File '{nama_file_sekarang}' telah diindeks ke database.")
    print(f"Total akumulasi data di database saat ini: {koleksi.count()} chunk.")

--- JALUR INGESTI DATA UNTUK: jurnal10.pdf ---
Memulai pemrosesan dokumen: jurnal10.pdf
Total halaman: 22
  -> [Halaman 1] Berhasil mengekstrak teks asli.
  -> [Halaman 2] Berhasil mengekstrak teks asli.
  -> [Halaman 3] Berhasil mengekstrak teks asli.
  -> [Halaman 4] Berhasil mengekstrak teks asli.
  -> [Halaman 5] Berhasil mengekstrak teks asli.
  -> [Halaman 6] Berhasil mengekstrak teks asli.
  -> [Halaman 7] Berhasil mengekstrak teks asli.
  -> [Halaman 8] Berhasil mengekstrak teks asli.
  -> [Halaman 9] Berhasil mengekstrak teks asli.
  -> [Halaman 10] Berhasil mengekstrak teks asli.
  -> [Halaman 11] Berhasil mengekstrak teks asli.
  -> [Halaman 12] Berhasil mengekstrak teks asli.
  -> [Halaman 13] Berhasil mengekstrak teks asli.
  -> [Halaman 14] Berhasil mengekstrak teks asli.
  -> [Halaman 15] Berhasil mengekstrak teks asli.
  -> [Halaman 16] Berhasil mengekstrak teks asli.
  -> [Halaman 17] Berhasil mengekstrak teks asli.
  -> [Halaman 18] Berhasil mengekstrak teks asli.
  -

# **5. Fungsi Pencarian Informasi Relevan**

In [ ]:
# =====================================================================
# SESI 5: RETRIEVAL (PENCARIAN CHUNK PALING RELEVAN)
# =====================================================================

def cari_chunk_relevan(pertanyaan, nama_sumber, k=3):
    """
    Mengambil k potongan teks paling relevan dari database ChromaDB.
    DIFILTER berdasarkan nama file agar tidak mengambil teks dari jurnal lain.
    """
    hasil = koleksi.query(
        query_texts=[pertanyaan],
        n_results=k,
        where={"sumber": nama_sumber} # PERBAIKAN: Filter Metadata
    )
    return hasil["documents"][0], hasil["metadatas"][0]

# =====================================================================
# UJI COBA PENCARIAN
# =====================================================================
# Pertanyaan standar untuk mengekstrak Gap Matrix
pertanyaan_pengguna = "Tolong ekstrak informasi Problem, Method, Dataset, Finding (Temuan), Limitation (Keterbatasan), dan Future Work dari paper ini."

# Eksekusi pencarian teks relevan (Gunakan nama_file_sekarang dari Sesi 4)
chunks, metadatas = cari_chunk_relevan(pertanyaan_pengguna, nama_file_sekarang, k=5)

print(f"Hasil pencarian data untuk: '{pertanyaan_pengguna}'\n")
print("================= DATA CHUNK RELEVAN =================")
for i, (chunk, meta) in enumerate(zip(chunks, metadatas)):
    sumber_file = meta.get("sumber", "Tidak diketahui")
    print(f"📌 [Rank {i+1}] File Asal: {sumber_file}")
    print("-" * 50)
    print(f"{chunk[:300]}...")
    print("-" * 50 + "\n")

Hasil pencarian data untuk: 'Tolong ekstrak informasi Problem, Method, Dataset, Finding (Temuan), Limitation (Keterbatasan), dan Future Work dari paper ini.'

================= DATA CHUNK RELEVAN =================
📌 [Rank 1] File Asal: jurnal10.pdf
--------------------------------------------------
C. Result and Discussion 
Data analysis in this study was assisted by the Jamovi application 
version 2.3.28. Data testing and analysis began with instrument testing, 
prerequisite testing, and hypothesis testing. The results of the analysis are 
presented below. 

1. Results 
The validity test resu...
--------------------------------------------------

📌 [Rank 2] File Asal: jurnal10.pdf
--------------------------------------------------
Figure 1. Research Design (Source: Sugiyono, 2011). 
The population and sample in this study were fourth-grade 
elementary school students in Sendang Subdistrict, Tulungagung Regency. 
The study used two learning groups, namely classes 4A and 4 B. Class 4A 


# **6. Integrasi LLM Universal dan Pengujian Metrik Performa**

In [ ]:
# =====================================================================
# SESI 6: GENERASI JAWABAN UNIVERSAL LLM + BENCHMARKING KINERJA
# =====================================================================
import time
import json
from openai import OpenAI

def jawab_dengan_rag_universal(pertanyaan, client_llm, nama_model, nama_sumber):
    # 1. Mengambil teks konteks
    chunk_relevan, _ = cari_chunk_relevan(pertanyaan, nama_sumber)
    konteks = "\n\n".join(chunk_relevan)

    # 2. Menyusun Prompt Akademik (PERBAIKAN PADA BAGIAN DATASET)
    prompt = f"""Anda adalah asisten peneliti yang ahli dalam analisis literatur.
Jawab pertanyaan berikut HANYA berdasarkan konteks yang diberikan.
Keluarkan hasilnya HANYA dalam format JSON yang valid, tanpa teks pembuka atau penutup, dengan skema berikut:
{{
  "judul": "Judul lengkap paper (jika tidak ada, tulis 'Tidak disebutkan')",
  "doi": "Nomor DOI paper (jika tidak ada, tulis 'Tidak disebutkan')",
  "problem": "Masalah yang diangkat dalam paper",
  "method": "Metode yang digunakan",
  "dataset": "Nama dataset. JIKA tidak ada nama spesifik (misal ImageNet), JELASKAN jenis/deskripsi datanya (misal: 'Data primer berupa 50 foto wajah siswa', 'Data transaksi perbankan'). Jangan hanya menulis 'Tidak disebutkan' jika ada penjelasan tentang data yang digunakan.",
  "finding": "Temuan utama",
  "limitation": "Keterbatasan penelitian",
  "future_work": "Saran penelitian selanjutnya (jika tidak ada, tulis 'Tidak disebutkan')"
}}

Konteks:
{konteks}

Pertanyaan: {pertanyaan}
Jawaban:"""

    print("=================== JAWABAN LLM ===================")
    waktu_mulai_request = time.perf_counter()

    # 3. Request ke API Server
    response_stream = client_llm.chat.completions.create(
        model=nama_model,
        messages=[{"role": "user", "content": prompt}],
        stream=True,
        stream_options={"include_usage": True},
        temperature=0.1
    )

    waktu_token_pertama = None
    jawaban_akhir = ""
    token_output_count = 0
    token_input_total = 0
    token_keseluruhan = 0

    # 4. Membaca Token Real-time
    for chunk in response_stream:
        if hasattr(chunk, 'usage') and chunk.usage is not None:
            token_input_total = chunk.usage.prompt_tokens
            token_keseluruhan = chunk.usage.total_tokens
            continue

        if chunk.choices and chunk.choices[0].delta.content:
            token_teks = chunk.choices[0].delta.content
            if waktu_token_pertama is None:
                waktu_token_pertama = time.perf_counter()
            jawaban_akhir += token_teks
            token_output_count += 1
            print(token_teks, end="", flush=True)

    waktu_selesai_generasi = time.perf_counter()
    print("\n" + "="*51)

    # 5. Penghitungan Latensi
    ttft_ms = (waktu_token_pertama - waktu_mulai_request) * 1000 if waktu_token_pertama else 0
    waktu_decode_detik = waktu_selesai_generasi - waktu_token_pertama if waktu_token_pertama else 0
    tps = token_output_count / waktu_decode_detik if waktu_decode_detik > 0 else 0

    if token_input_total == 0:
        token_input_total = len(prompt) // 4
        token_keseluruhan = token_input_total + token_output_count

    # 6. Laporan Statistik
    print("\n📊 STATISTIK KINERJA & BIAYA TOKEN:")
    print(f"⏱️ Time to First Token (TTFT) : {ttft_ms:.2f} ms")
    print(f"⚡ Throughput (TPS)           : {tps:.2f} tokens/sec")
    print(f"📥 Token Input (Prompt)       : {token_input_total} token")
    print(f"📤 Token Output (Completion)  : {token_output_count} token")
    print(f"🔄 Total Token Terpakai       : {token_keseluruhan} token")

    biaya_estimasi = ((token_input_total * 0.15) + (token_output_count * 0.60)) / 1_000_000
    print(f"💰 Estimasi Biaya Sesi Ini    : ${biaya_estimasi:.7f} USD")

    # 7. Mengemas metrik
    metrik_performa = {
        "ttft_ms": ttft_ms,
        "tps": tps,
        "token_input": token_input_total,
        "token_output": token_output_count,
        "total_token": token_keseluruhan,
        "biaya_usd": biaya_estimasi
    }

    return jawaban_akhir, konteks, metrik_performa


# =====================================================================
# EKSEKUSI PENGUJIAN
# =====================================================================
api_key_kamu = "MASUKAN API KEY DISINI"
client_pilihan = OpenAI(api_key=api_key_kamu, base_url="https://api.groq.com/openai/v1")
model_pilihan = "openai/gpt-oss-120b"

if 'pertanyaan_pengguna' in locals():
    jawaban, konteks_terpilih, metrik_terakhir = jawab_dengan_rag_universal(
        pertanyaan=pertanyaan_pengguna,
        client_llm=client_pilihan,
        nama_model=model_pilihan,
        nama_sumber=nama_file_sekarang
    )
else:
    print("⚠️ Eror: Variabel 'pertanyaan_pengguna' tidak ditemukan.")

=================== JAWABAN LLM ===================
{
  "judul": "Tidak disebutkan",
  "doi": "10.26811/didaktika.v9i3.1997",
  "problem": "Menilai efektivitas perlakuan pembelajaran terhadap kemampuan mengukur panjang pada siswa kelas empat SD di Kecamatan Sendang, Kabupaten Tulungagung.",
  "method": "Desain eksperimental dengan dua kelompok belajar (kelas 4A sebagai kelompok eksperimen dan kelas 4B sebagai kelompok kontrol). Pengambilan sampel nonprobabilistik dengan teknik purposive sampling. Instrumen berupa tes tertulis 20 soal pilihan ganda (level kognitif C1‑C6) diberikan secara pre‑test dan post‑test. Validitas instrumen diuji dengan exploratory factor analysis (EFA) dan konstruksi validitas; reliabilitas diuji dengan Cronbach’s Alpha. Analisis data dilakukan menggunakan Jamovi 2.3.28 meliputi uji asumsi (Shapiro‑Wilk untuk normalitas, uji homogenitas), serta uji hipotesis untuk membandingkan hasil pre‑ dan post‑test antara kelompok eksperimen dan kontrol.",
  "dataset": "Data

# **7. Logging Pengujian & Ekspor Data (Excel/CSV) untuk Bab Analisis Dokumen**

In [ ]:
# =====================================================================
# SESI 7: AUTOMATED LOGGING & EKSPOR DATA KE GOOGLE DRIVE
# =====================================================================
import pandas as pd
import datetime
import os
import json

def simpan_log_pengujian(metrik_data, folder_tujuan):
    variabel_wajib = ['pertanyaan_pengguna', 'jawaban', 'konteks_terpilih', 'model_pilihan']
    for var in variabel_wajib:
        if var not in globals():
            print(f"⚠️ Eror: Variabel '{var}' tidak ditemukan. Pastikan Sesi 5 dan Sesi 6 sudah dijalankan.")
            return

    # Agar nama file baru tidak menumpuk dengan data percobaan lama
    path_file_excel = os.path.join(folder_tujuan, "log_pengujian_rag_v2.xlsx")
    path_file_csv = os.path.join(folder_tujuan, "log_pengujian_rag_v2.csv")

    waktu_sekarang = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # Parsing JSON
    try:
        json_str = jawaban.strip().removeprefix('```json').removesuffix('```').strip()
        parsed_jawaban = json.loads(json_str)
    except Exception as e:
        print(f"⚠️ Peringatan: Gagal parsing JSON. Error: {e}")
        parsed_jawaban = {
            "judul": "Gagal Parse", "doi": "Gagal Parse",
            "problem": "Gagal Parse", "method": "Gagal Parse", "dataset": "Gagal Parse",
            "finding": jawaban, "limitation": "Gagal Parse", "future_work": "Gagal Parse"
        }

    # Menyusun struktur baris data baru (PENAMBAHAN JUDUL & DOI)
    data_baru = {
        "Waktu Pengujian": [waktu_sekarang],
        "Model LLM": [model_pilihan],
        "Nama File": [nama_file_sekarang],
        "Judul": [parsed_jawaban.get("judul", "")],
        "DOI": [parsed_jawaban.get("doi", "")],
        "Problem": [parsed_jawaban.get("problem", "")],
        "Method": [parsed_jawaban.get("method", "")],
        "Dataset": [parsed_jawaban.get("dataset", "")],
        "Temuan": [parsed_jawaban.get("finding", "")],
        "Keterbatasan": [parsed_jawaban.get("limitation", "")],
        "Future Work": [parsed_jawaban.get("future_work", "")],
        "Konteks RAG": [konteks_terpilih],
        "TTFT (ms)": [round(metrik_data.get('ttft_ms', 0), 2)],
        "Throughput (TPS)": [round(metrik_data.get('tps', 0), 2)],
        "Token Input": [metrik_data.get('token_input', 0)],
        "Token Output": [metrik_data.get('token_output', 0)],
        "Total Token": [metrik_data.get('total_token', 0)],
        "Estimasi Biaya (USD)": [metrik_data.get('biaya_usd', 0)]
    }

    df_baru = pd.DataFrame(data_baru)

    # Simpan (kumulatif)
    if os.path.exists(path_file_excel):
        df_lama = pd.read_excel(path_file_excel)
        df_total = pd.concat([df_lama, df_baru], ignore_index=True)
    else:
        df_total = df_baru

    if os.path.exists(path_file_csv):
        df_baru.to_csv(path_file_csv, mode='a', header=False, index=False)
    else:
        df_baru.to_csv(path_file_csv, mode='w', header=True, index=False)

    df_total.to_excel(path_file_excel, index=False)

    print("=================== LOGGING SYSTEM ===================")
    print(f"🎉 SUKSES! Data Jurnal {nama_file_sekarang} berhasil dicatat.")
    print(f"📁 File Excel tersimpan di: {path_file_excel}")
    print(f"📊 Total baris saat ini: {len(df_total)} data.")
    print("======================================================")

    print("\nPreview Data Gap Matrix (3 Baris Terakhir):")
    # Menampilkan kolom penting termasuk DOI
    display(df_total[["Nama File", "DOI", "Problem", "Method"]].tail(3))

# --- EKSEKUSI PENCATATAN LOG ---
simpan_log_pengujian(metrik_data=metrik_terakhir, folder_tujuan=folder_output)

=================== LOGGING SYSTEM ===================
🎉 SUKSES! Data Jurnal jurnal10.pdf berhasil dicatat.
📁 File Excel tersimpan di: /content/drive/MyDrive/Tugas_RAG_GapMatrix/log_pengujian_rag_v2.xlsx
📊 Total baris saat ini: 10 data.

Preview Data Gap Matrix (3 Baris Terakhir):


,Nama File,DOI,Problem,Method
7,jurnal08.pdf,10.3390/math14173026,Bagaimana memilih contoh pelatihan sintetik be...,Membangun pipeline berlapis: (1) augmentasi da...
8,jurnal09.pdf,Tidak disebutkan,Masalah membaca praktis yang dihadapi siswa di...,Penelitian tindakan kelas (classroom action re...
9,jurnal10.pdf,10.26811/didaktika.v9i3.1997,Menilai efektivitas perlakuan pembelajaran ter...,Desain eksperimental dengan dua kelompok belaj...
